# Chess Policy Model — Dataset Exploration

This notebook explores the Lichess chess position evaluation dataset used to train the ChessMind Arena policy model.

## Objective

We want to transform:

    FEN + Stockfish principal variation

into supervised chess training examples:

    Chess position → strong candidate move

The first stage is dataset exploration only.

We will:

1. Access the Lichess evaluation dataset.
2. Inspect its schema.
3. Examine FEN positions.
4. Examine Stockfish principal variations.
5. Extract the first move from the principal variation.
6. Validate that the move is legal using python-chess.
7. Inspect evaluation and search depth.
8. Determine suitable filtering criteria for our training dataset.

We will NOT download the complete dataset.

In [1]:
!pip install -q datasets python-chess pandas matplotlib

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.1/6.1 MB 50.9 MB/s eta 0:00:0000:0100:01
  Preparing metadata (setup.py) ... done


In [2]:
import chess
import pandas as pd
from datasets import load_dataset

In [3]:
dataset = load_dataset(
    "Lichess/chess-position-evaluations",
    split="train",
    streaming=True,
)

/usr/local/lib/python3.13/dist-packages/huggingface_hub/utils/_auth.py:138: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
  warnings.warn(f"\nError while fetching `HF_TOKEN` secret value from your vault: '{str(e)}'.")


README.md:   0%|          | 0.00/1.80k [00:00<?, ?B/s]

Resolving data files:   0%|          | 0/20 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/20 [00:00<?, ?it/s]

In [4]:
samples = []

for i, row in enumerate(dataset):
    samples.append(row)

    if i == 9:
        break

In [6]:
print(samples[0])

{'fen': '7r/1p3k2/p1bPR3/5p2/2B2P1p/8/PP4P1/3K4 b - -', 'line': 'f7g7 e6e2 h8d8 e2d2 b7b5 c4b3 g7f6 d1e1 a6a5 a2a3', 'depth': 46, 'knodes': 4189972, 'cp': 69, 'mate': None}


In [7]:
df = pd.DataFrame(samples)

In [9]:
print(df.info())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 10 entries, 0 to 9
Data columns (total 6 columns):
 #   Column  Non-Null Count  Dtype 
---  ------  --------------  ----- 
 0   fen     10 non-null     object
 1   line    10 non-null     object
 2   depth   10 non-null     int64 
 3   knodes  10 non-null     int64 
 4   cp      10 non-null     int64 
 5   mate    0 non-null      object
dtypes: int64(3), object(3)
memory usage: 612.0+ bytes
None


In [10]:
row = df.iloc[0]

print("FEN:")
print(row["fen"])

print("\nPrincipal Variation:")
print(row["line"])

print("\nDepth:")
print(row["depth"])

print("\nNodes:")
print(row["knodes"])

print("\nCentipawn Evaluation:")
print(row["cp"])

print("\nMate:")
print(row["mate"])

FEN:
7r/1p3k2/p1bPR3/5p2/2B2P1p/8/PP4P1/3K4 b - -

Principal Variation:
f7g7 e6e2 h8d8 e2d2 b7b5 c4b3 g7f6 d1e1 a6a5 a2a3

Depth:
46

Nodes:
4189972

Centipawn Evaluation:
69

Mate:
None


In [11]:
board = chess.Board(row["fen"])

print(board)

. . . . . . . r
. p . . . k . .
p . b P R . . .
. . . . . p . .
. . B . . P . p
. . . . . . . .
P P . . . . P .
. . . K . . . .


In [15]:
pv = row["line"].split()

print(pv)

['f7g7', 'e6e2', 'h8d8', 'e2d2', 'b7b5', 'c4b3', 'g7f6', 'd1e1', 'a6a5', 'a2a3']


In [16]:
target_move = pv[0]

print("\nTarget Move:", target_move)


Target Move: f7g7


In [17]:
move = chess.Move.from_uci(target_move)

print("\nTarget move:", target_move)
print("Legal move:", move in board.legal_moves)


Target move: f7g7
Legal move: True


In [18]:
board.push(move)

print("\nBoard after move:")
print(board)


Board after move:
. . . . . . . r
. p . . . . k .
p . b P R . . .
. . . . . p . .
. . B . . P . p
. . . . . . . .
P P . . . . P .
. . . K . . . .


In [19]:
print("CP:", row["cp"])
print("Mate:", row["mate"])

CP: 69
Mate: None


In [20]:
original_board = chess.Board(row["fen"])

print("Turn:", "White" if original_board.turn else "Black")

Turn: Black


In [21]:
df[["depth", "knodes", "cp", "mate"]]

,depth,knodes,cp,mate
0,46,4189972,69,None
1,46,4189972,163,None
2,46,4189972,229,None
3,46,4189972,231,None
4,46,4189972,237,None
5,58,491568,0,None
6,58,491568,0,None
7,57,1176702,0,None
8,57,1176702,0,None
9,57,1176702,0,None


In [22]:
print("Average depth:", df["depth"].mean())
print("Minimum depth:", df["depth"].min())
print("Maximum depth:", df["depth"].max())

Average depth: 51.7
Minimum depth: 46
Maximum depth: 58


In [23]:
print("Mate positions:", df["mate"].notna().sum())
print("Normal evaluations:", df["cp"].notna().sum())

Mate positions: 0
Normal evaluations: 10


In [24]:
def is_valid_target(row):
    try:
        board = chess.Board(row["fen"])
        moves = row["line"].split()

        if not moves:
            return False

        move = chess.Move.from_uci(moves[0])

        return move in board.legal_moves

    except Exception:
        return False

In [25]:
df["valid_target"] = df.apply(is_valid_target, axis=1)

df["valid_target"].value_counts()

,count
valid_target,
True,10


In [26]:
example = {
    "fen": row["fen"],
    "target_move": row["line"].split()[0],
    "cp": row["cp"],
    "mate": row["mate"],
    "depth": row["depth"],
}

example

{'fen': '7r/1p3k2/p1bPR3/5p2/2B2P1p/8/PP4P1/3K4 b - -',
 'target_move': 'f7g7',
 'cp': np.int64(69),
 'mate': None,
 'depth': np.int64(46)}